<a href="https://colab.research.google.com/github/smozwald/foundation-flood/blob/main/notebooks/03_initial_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase One Summary & Phase Two Planning

This notebook covers:
1. **Phase One status** — what we built, what data we have
2. **AI4GOOD / alternative flood mapping** — comparing with our current SAR Otsu approach
3. **Current data analysis** — strengths and weaknesses of the pixel approach
4. **Flood event visualisations** — mapped flooded vs unflooded agricultural pixels per event
5. **Fields of the World** — assessing a field-level alternative to pixel sampling
6. **Phase Two planning** — inundation method comparison and decision point

In [ ]:
# CELL 1 — Install dependencies
!pip install psycopg2-binary python-dotenv pandas matplotlib shapely geopandas contextily earthengine-api folium pillow reportlab -q

In [ ]:
# CELL 2 — Imports & Supabase connection
import os
import io
import json
import base64
import urllib.request
from datetime import datetime, timedelta

import psycopg2
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.gridspec as gridspec
from PIL import Image

from google.colab import userdata

CONN_STRING = userdata.get('SUPABASE_CONN_STRING')

def db():
    return psycopg2.connect(CONN_STRING)

# Quick smoke-test
with db() as conn:
    cur = conn.cursor()
    cur.execute('SELECT COUNT(*) FROM pixels_static')
    n = cur.fetchone()[0]
print(f'Connected OK — {n:,} pixels in pixels_static')

In [ ]:
# CELL 3 — GEE auth
import ee
try:
    ee.Initialize(project='foundation-flood')
    print('GEE already initialised')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='foundation-flood')
    print('GEE initialised')

---
## 1. Phase One Status

In [ ]:
# CELL 4 — Phase One row-count summary
queries = {
    'discharge_stations':   'SELECT COUNT(*) FROM discharge_stations',
    'discharge_ts':         'SELECT COUNT(*) FROM discharge_ts',
    'flood_events':         'SELECT COUNT(*) FROM flood_events',
    'study_zones':          'SELECT COUNT(*) FROM study_zones',
    'zone_flood_analysis (SUCCESS)': """
        SELECT COUNT(*) FROM zone_flood_analysis WHERE status = 'SUCCESS'""",
    'pixels_static':        'SELECT COUNT(*) FROM pixels_static',
    'ts_sentinel1':         'SELECT COUNT(*) FROM ts_sentinel1',
    'pixel_flooded':        'SELECT COUNT(*) FROM pixel_flooded',
}

rows = []
with db() as conn:
    cur = conn.cursor()
    for label, sql in queries.items():
        cur.execute(sql)
        rows.append({'Table': label, 'Rows': cur.fetchone()[0]})

status_df = pd.DataFrame(rows)
status_df['Status'] = status_df.apply(
    lambda r: '✓ complete' if r['Rows'] > 0 else '— empty', axis=1
)
print(status_df.to_string(index=False))

In [ ]:
# CELL 5 — Zone 000099_initial breakdown
ZONE_PATTERN = '000099_initial'

with db() as conn:
    zone_df = pd.read_sql("""
        SELECT
            zfa.zone_id,
            fe.flood_start,
            fe.flood_end,
            fe.max_category,
            zfa.status,
            zfa.percentage_flooded,
            zfa.scene_date
        FROM zone_flood_analysis zfa
        JOIN flood_events fe ON fe.event_id = zfa.event_id
        WHERE zfa.zone_id ~ %(pat)s
        ORDER BY fe.flood_start
    """, conn, params={'pat': ZONE_PATTERN})

    pix_df = pd.read_sql("""
        SELECT
            ps.pixel_id,
            ST_Y(ps.geom::geometry) AS lat,
            ST_X(ps.geom::geometry) AS lon,
            ps.elevation,
            ps.slope,
            ps.dist_to_river_m
        FROM pixels_static ps
        WHERE ps.zone_id ~ %(pat)s
    """, conn, params={'pat': ZONE_PATTERN})

    flooded_df = pd.read_sql("""
        SELECT
            pf.pixel_id,
            pf.event_id,
            pf.flooded,
            pf.vv_flood,
            pf.vh_flood
        FROM pixel_flooded pf
        JOIN pixels_static ps ON ps.pixel_id = pf.pixel_id
        WHERE ps.zone_id ~ %(pat)s
    """, conn, params={'pat': ZONE_PATTERN})

print(f'Zone events: {len(zone_df)}  |  Pixels: {len(pix_df):,}  |  Pixel-flood labels: {len(flooded_df):,}')
print()
print(zone_df[['zone_id','flood_start','max_category','status','percentage_flooded','scene_date']].to_string(index=False))

---
## 2. AI4GOOD & Alternative Flood Mapping Approaches

### What we built (Phase One)
We use Sentinel-1 SAR backscatter with an **Otsu threshold** to detect open water per zone per event. The threshold is computed on a VV-polarisation composite clipped to the zone, and pixels below the threshold are labelled flooded. Valley pixels (low-lying non-agricultural areas) are excluded.

**Strengths:**
- SAR penetrates cloud cover — critical during active monsoon floods
- Otsu is unsupervised and adapts to each scene
- GEE allows processing at scale without local compute

**Weaknesses:**
- Otsu can mis-classify smooth bare soil or dense crops as water
- SAR speckling adds noise at the pixel level
- Requires a good flood-free reference scene (temporal change detection would improve this)
- Revisit time (~6 days) means some short flood events are missed

### Alternatives

| Approach | Pros | Cons |
|----------|------|------|
| **JRC Global Surface Water** (Landsat) | Long record (1984–), monthly | Optical only — cloud-blocked during floods |
| **Copernicus EMS** | Human-validated, per-event maps | Sparse coverage, few Pakistan events |
| **AI4GOOD FloodAI** | DL model, uses S1+S2 fusion | Requires GPU, not always accessible |
| **Google Flood Hub** | Real-time, global inundation maps | API access limited, no historical per-pixel labels |
| **MODIS NRT Flood Product** (LANCE) | Near-real-time, free | 500 m resolution — too coarse for field level |
| **UNOSAT** | Validated disaster maps | Only major declared disasters |
| **Our SAR Otsu** | Cloud-free, adaptive, scriptable | See weaknesses above |

### AI4GOOD FloodAI note
The [AI4GOOD Lab FloodAI](https://floodai.ca/) project trains neural networks on fused S1+S2 imagery. Their models are available as checkpoints but require significant GPU compute. For historical ground-truth comparison in our zone, we could:
1. Pull their inference outputs for 2022–2025 Pakistan events if available via their API
2. Use JRC GSW monthly maps as an optical cross-check for events in dry months
3. Use Copernicus EMS activation maps for the Aug 2022 and Aug 2025 Pakistan floods (both are major activations)

**Recommendation:** For Phase Two ground truth validation, use **Copernicus EMS** maps for the 2022 and 2025 events (these are freely downloadable shapefiles). For the remaining events, our SAR Otsu labels are likely the best available option at this spatial resolution.

---
## 3. Current Data Analysis — Strengths & Weaknesses

In [ ]:
# CELL 6 — Per-event flood rate from pixel_flooded
if len(flooded_df) == 0:
    print('pixel_flooded is empty — run ingest_s1_flood_pixels.py first.')
    print('Showing zone_flood_analysis percentage_flooded as proxy:')
    success = zone_df[zone_df['status'] == 'SUCCESS'].copy()
    success['flood_start'] = pd.to_datetime(success['flood_start'])
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = {1: '#fee8c8', 2: '#fdbb84', 3: '#fc8d59', 4: '#e34a33', 5: '#b30000'}
    bars = ax.bar(
        range(len(success)),
        success['percentage_flooded'],
        color=[colors.get(int(c), '#aaa') for c in success['max_category']]
    )
    ax.set_xticks(range(len(success)))
    ax.set_xticklabels(
        [f"{row['flood_start'].strftime('%b %Y')}\n(cat {int(row['max_category'])})" for _, row in success.iterrows()],
        fontsize=8
    )
    ax.set_ylabel('% zone flooded (Otsu SAR)')
    ax.set_title('Flood extent per event — zone 000099_initial (Chenab)')
    patches = [mpatches.Patch(color=colors[c], label=f'Category {c}') for c in range(1, 6)]
    ax.legend(handles=patches, loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.savefig('temp_images/flood_extent_by_event.png', dpi=150)
    plt.show()
else:
    # pixel_flooded is populated
    summary = (
        flooded_df.groupby('event_id')['flooded']
        .agg(total='count', flooded_n='sum')
        .assign(pct_flooded=lambda d: 100 * d['flooded_n'] / d['total'])
        .reset_index()
        .merge(zone_df[['zone_id','flood_start','max_category']].assign(
            flood_start=lambda d: pd.to_datetime(d['flood_start'])
        ), left_on='event_id', right_index=True, how='left')
    )
    print(summary[['flood_start','max_category','total','flooded_n','pct_flooded']].to_string(index=False))

In [ ]:
# CELL 7 — Pixel terrain distribution
if len(pix_df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))

    axes[0].hist(pix_df['elevation'].dropna(), bins=40, color='steelblue', edgecolor='white')
    axes[0].set_xlabel('Elevation (m)')
    axes[0].set_title('Pixel elevation')

    axes[1].hist(pix_df['slope'].dropna(), bins=40, color='darkorange', edgecolor='white')
    axes[1].set_xlabel('Slope (°)')
    axes[1].set_title('Pixel slope')

    axes[2].hist(pix_df['dist_to_river_m'].dropna(), bins=40, color='seagreen', edgecolor='white')
    axes[2].set_xlabel('Distance to river (m)')
    axes[2].set_title('Distance to river')

    plt.suptitle('pixels_static terrain metrics — zone 000099_initial', fontsize=10)
    plt.tight_layout()
    plt.savefig('temp_images/pixel_terrain_distribution.png', dpi=150)
    plt.show()

    print(pix_df[['elevation','slope','dist_to_river_m']].describe().round(1))
else:
    print('No pixels found for zone pattern:', ZONE_PATTERN)

### Strengths of the current pixel approach

- **Scale**: ~49,000 pixels across 10 study zones gives a decent training set
- **Feature richness**: each pixel has elevation, slope, distance to river, and (once `ingest_s1_flood_pixels.py` runs) VV/VH time series
- **Agri-specific**: pixels are sampled from ESA WorldCover cropland class only
- **Multi-event labels**: up to 64 labelled events per zone enables temporal learning

### Weaknesses

- **Spatial resolution mismatch**: 10 m pixels don't respect field boundaries — a pixel may straddle a field edge or ditch
- **No field identity**: we can't compute per-field inundation statistics or duration without field boundaries
- **Scalability**: the GEE pixel sampling and S1 export pipeline is slow and manual per zone. Adding new rivers requires re-running all 5 steps with careful parameterisation
- **Label noise**: the Otsu-derived `flooded` label propagates any SAR mis-classification into the training signal
- **No duration signal**: we know if a pixel was flooded at the S1 acquisition date, not how long it stayed flooded

---
## 4. Flood Event Visualisations

In [ ]:
# CELL 8 — Fetch S2 RGB thumbnail from GEE for a zone centred on pixel cloud
import os
os.makedirs('temp_images', exist_ok=True)

def fetch_s2_thumbnail(lat_center, lon_center, date_str, buffer_m=5000, width=512):
    """Return PIL Image of S2 RGB around (lat, lon) near date_str."""
    pt = ee.Geometry.Point([lon_center, lat_center])
    region = pt.buffer(buffer_m).bounds()
    date = ee.Date(date_str)
    s2 = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(region)
        .filterDate(date.advance(-60, 'day'), date.advance(60, 'day'))
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
        .sort('CLOUDY_PIXEL_PERCENTAGE')
        .first()
        .select(['B4', 'B3', 'B2'])
    )
    url = s2.getThumbURL({
        'region': region,
        'dimensions': width,
        'format': 'png',
        'min': 0,
        'max': 3000,
        'gamma': 1.4,
    })
    with urllib.request.urlopen(url) as resp:
        img = Image.open(io.BytesIO(resp.read()))
    return img, region.getInfo()

print('S2 thumbnail helper defined')

In [ ]:
# CELL 9 — Plot flooded vs unflooded pixels per event on S2 basemap
# Requires pixel_flooded to be populated. Falls back to pixel scatter if empty.

success_events = zone_df[zone_df['status'] == 'SUCCESS'].copy()
success_events['flood_start'] = pd.to_datetime(success_events['flood_start'])
success_events = success_events.reset_index(drop=True)

if len(pix_df) == 0:
    print('No pixel data — skipping visualisation')
else:
    lat_c = pix_df['lat'].mean()
    lon_c = pix_df['lon'].mean()

    # Try to get S2 basemap using the first event date
    ref_date = success_events.iloc[0]['flood_start'].strftime('%Y-%m-%d') if len(success_events) > 0 else '2023-01-01'
    try:
        bg_img, region_info = fetch_s2_thumbnail(lat_c, lon_c, ref_date)
        # Extract bounding box for extent mapping
        coords = region_info['coordinates'][0]
        lon_min = min(c[0] for c in coords)
        lon_max = max(c[0] for c in coords)
        lat_min = min(c[1] for c in coords)
        lat_max = max(c[1] for c in coords)
        has_bg = True
        print(f'S2 basemap loaded ({lon_min:.4f},{lat_min:.4f}) → ({lon_max:.4f},{lat_max:.4f})')
    except Exception as e:
        print(f'S2 basemap unavailable ({e}) — plotting without background')
        has_bg = False
        lon_min, lon_max = pix_df['lon'].min() - 0.01, pix_df['lon'].max() + 0.01
        lat_min, lat_max = pix_df['lat'].min() - 0.01, pix_df['lat'].max() + 0.01

    n_events = len(success_events)
    ncols = min(4, n_events)
    nrows = (n_events + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows))
    if n_events == 1:
        axes = [[axes]]
    elif nrows == 1:
        axes = [axes]

    for idx, (_, ev) in enumerate(success_events.iterrows()):
        row, col = divmod(idx, ncols)
        ax = axes[row][col]

        if has_bg:
            ax.imshow(bg_img, extent=[lon_min, lon_max, lat_min, lat_max], aspect='auto', alpha=0.6)

        # Get per-event pixel labels
        ev_flooded = flooded_df[flooded_df['event_id'] == ev.get('event_id', -1)] if len(flooded_df) > 0 else pd.DataFrame()

        if len(ev_flooded) > 0:
            merged = pix_df.merge(ev_flooded[['pixel_id','flooded']], on='pixel_id', how='left')
            merged['flooded'] = merged['flooded'].fillna(False)
            unflooded = merged[~merged['flooded']]
            flooded = merged[merged['flooded']]
            ax.scatter(unflooded['lon'], unflooded['lat'], s=2, c='#d73027', alpha=0.6, label='Unflooded agri')
            ax.scatter(flooded['lon'], flooded['lat'], s=2, c='#4575b4', alpha=0.8, label='Flooded agri')
            pct = ev['percentage_flooded']
        else:
            # No pixel labels — show all pixels grey and note zone-level pct
            ax.scatter(pix_df['lon'], pix_df['lat'], s=2, c='#999999', alpha=0.5, label='Agri pixels')
            pct = ev['percentage_flooded']

        scene = ev['scene_date'].strftime('%Y-%m-%d') if pd.notnull(ev.get('scene_date')) else str(ev['flood_start'].date())
        ax.set_title(
            f"{ev['flood_start'].strftime('%b %Y')} | Cat {int(ev['max_category'])}\n"
            f"Scene: {scene} | {pct:.1f}% flooded",
            fontsize=8
        )
        ax.set_xlim(lon_min, lon_max)
        ax.set_ylim(lat_min, lat_max)
        ax.tick_params(labelsize=7)
        if idx == 0:
            ax.legend(loc='upper right', fontsize=6, markerscale=4)

    # Hide unused axes
    for idx in range(n_events, nrows * ncols):
        row, col = divmod(idx, ncols)
        axes[row][col].set_visible(False)

    plt.suptitle('Flood event maps — zone 000099_initial (Chenab River, Pakistan)', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig('temp_images/flood_events_map.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved to temp_images/flood_events_map.png')

In [ ]:
# CELL 10 — Per-event S2 basemaps (one per event, higher quality)
# Only run if pixel_flooded is populated and you want per-event satellite backgrounds
RENDER_PER_EVENT = False  # Set True to download individual S2 thumbnails per event (slow)

if RENDER_PER_EVENT and len(pix_df) > 0:
    lat_c = pix_df['lat'].mean()
    lon_c = pix_df['lon'].mean()

    for idx, (_, ev) in enumerate(success_events.iterrows()):
        date_str = ev['flood_start'].strftime('%Y-%m-%d')
        try:
            bg, region_info = fetch_s2_thumbnail(lat_c, lon_c, date_str)
        except Exception as e:
            print(f'  Skip {date_str}: {e}')
            continue

        coords = region_info['coordinates'][0]
        lon_min_ = min(c[0] for c in coords)
        lon_max_ = max(c[0] for c in coords)
        lat_min_ = min(c[1] for c in coords)
        lat_max_ = max(c[1] for c in coords)

        ev_flooded = flooded_df[flooded_df['event_id'] == ev.get('event_id', -1)] if len(flooded_df) > 0 else pd.DataFrame()

        fig, ax = plt.subplots(figsize=(7, 6))
        ax.imshow(bg, extent=[lon_min_, lon_max_, lat_min_, lat_max_], aspect='auto', alpha=0.65)

        if len(ev_flooded) > 0:
            merged = pix_df.merge(ev_flooded[['pixel_id','flooded']], on='pixel_id', how='left')
            merged['flooded'] = merged['flooded'].fillna(False)
            ax.scatter(merged.loc[~merged['flooded'],'lon'], merged.loc[~merged['flooded'],'lat'],
                       s=3, c='#d73027', alpha=0.7, label='Unflooded agri')
            ax.scatter(merged.loc[merged['flooded'],'lon'], merged.loc[merged['flooded'],'lat'],
                       s=3, c='#4575b4', alpha=0.85, label='Flooded agri')
        else:
            ax.scatter(pix_df['lon'], pix_df['lat'], s=3, c='#999', alpha=0.5)

        scene = ev['scene_date'].strftime('%Y-%m-%d') if pd.notnull(ev.get('scene_date')) else date_str
        ax.set_title(
            f"{ev['flood_start'].strftime('%B %Y')} | Category {int(ev['max_category'])}\n"
            f"Scene: {scene} | {ev['percentage_flooded']:.1f}% flooded (zone-level Otsu)",
            fontsize=9
        )
        ax.legend(loc='upper right', fontsize=8, markerscale=3)
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        plt.tight_layout()
        fname = f'temp_images/event_{idx:02d}_{date_str}.png'
        plt.savefig(fname, dpi=150)
        plt.show()
        print(f'Saved {fname}')
else:
    print('Set RENDER_PER_EVENT = True to download per-event satellite basemaps')

---
## 5. Fields of the World — Feasibility Assessment

### What is Fields of the World?

[Fields of the World (FoTW)](https://fieldsofthe.world/) is a global benchmark dataset of agricultural field boundaries derived from crowdsourced mapping and ML inference. The 2025 basemap provides polygon boundaries for fields worldwide, with a confidence score per field.

**Access options:**
- GEE asset: `projects/sat-io/open-datasets/FIELDS-OF-THE-WORLD` (check availability)
- Direct download: country-level GeoPackage files from the FoTW website
- Google Drive shared datasets

We will use fields with **confidence ≥ 70%** (2025 basemap), filtered to our zone 000099_initial bounding box.

### Pros vs Pixel approach

| Factor | Pixel (current) | Fields of the World |
|--------|----------------|--------------------|
| Spatial unit | 10 m raster pixel | Field polygon (variable size) |
| Boundary respect | No — straddles edges | Yes — flood % per whole field |
| Interpretability | Hard (49k isolated points) | High — field-level decision |
| Completeness | ESA WorldCover coverage | FoTW coverage (may have gaps) |
| Inundation duration | Single date per event | Can compute A–B days if multi-date |
| Scalability | Slow (GEE per-pixel export) | Faster (zonal stats per field) |
| Training signal | Clean grid, easy to batch | Irregular polygons, need zonal aggregation |
| Model type suitability | CNN / tabular | Graph / tabular / polygon-aware CNN |

**Verdict:** Field-level is more interpretable and actionable ("field X will flood") but adds complexity in zonal aggregation and requires FoTW to have good coverage in Pakistan. The pixel approach is simpler to train but produces outputs that are harder to use in practice.

In [ ]:
# CELL 11 — Load Fields of the World for zone 000099_initial (GEE)
# Check if FoTW is available as a GEE asset
if len(pix_df) == 0:
    print('No pixel data — skipping FoTW comparison')
else:
    lat_c = pix_df['lat'].mean()
    lon_c = pix_df['lon'].mean()
    buf = 0.04  # ~4.5 km
    zone_bbox = ee.Geometry.BBox(lon_c - buf, lat_c - buf, lon_c + buf, lat_c + buf)

    FOTW_ASSET = 'projects/sat-io/open-datasets/FIELDS-OF-THE-WORLD/FotW_2025'
    try:
        fotw = ee.FeatureCollection(FOTW_ASSET).filterBounds(zone_bbox)
        n_fields = fotw.size().getInfo()
        print(f'FoTW fields in zone: {n_fields}')
    except Exception as e:
        print(f'FoTW GEE asset not found or access denied: {e}')
        print('Alternative: download Pakistan GeoPackage from https://fieldsofthe.world/download')
        n_fields = 0

    if n_fields > 0:
        # Filter to >70% confidence (field 'confidence' or 'score' — check schema)
        try:
            high_conf = fotw.filter(ee.Filter.gte('confidence', 0.70))
            n_high = high_conf.size().getInfo()
            print(f'Fields with confidence >= 0.70: {n_high}')
        except Exception:
            high_conf = fotw  # attribute name may differ
            n_high = n_fields
            print('Could not filter by confidence — check field attribute name in FoTW schema')

In [ ]:
# CELL 12 — Compute per-field flood % per event (if FoTW loaded)
# For each field polygon, intersect with zone_flood_analysis SAR flood mask and
# compute percentage of field pixels classified as flooded.
# Also compute inundation duration using the GEE S1 time series.

# Duration bins (days from flood start):
DURATION_BINS = [(0, 4), (4, 8), (8, 16), (16, 32)]

def compute_field_flood_pct(field_fc, zone_id, event_id, scene_date, conn_string):
    """
    Placeholder — in production this would:
    1. Load the SAR flood mask raster from GEE for this zone+event
    2. Reduce each field polygon to a mean flood value
    3. Return a GeoDataFrame with field_id, pct_flooded, inundation_days
    """
    raise NotImplementedError(
        'Implement after deciding: use GEE zonal stats (reduceRegions) '
        'on the SAR flood mask, or load pixel_flooded labels and spatially join.'
    )

print('Per-field flood % function stubbed — implement after FoTW access confirmed.')
print()
print('Inundation duration approach:')
print('  For each S1 acquisition within flood_start → flood_end + 12d:')
print('  If pixel/field is flooded at T1 but not T2 (next acquisition), count T1→T2 as flooded.')
print('  Sum across acquisitions to get total flooded days.')
print()
for a, b in DURATION_BINS:
    print(f'  Bin: {a}–{b} days flooded')

In [ ]:
# CELL 13 — Side-by-side: current pixels vs FoTW fields extent
if len(pix_df) > 0 and 'n_fields' in dir() and n_fields > 0:
    # Pull FoTW field centroids for plotting
    try:
        fotw_sample = high_conf.limit(500)
        fotw_pts = fotw_sample.map(
            lambda f: f.set({'lon': f.geometry().centroid().coordinates().get(0),
                             'lat': f.geometry().centroid().coordinates().get(1)})
        )
        fotw_pd = pd.DataFrame(fotw_pts.getInfo()['features'])
        fotw_pd['lon'] = fotw_pd['properties'].apply(lambda p: p['lon'])
        fotw_pd['lat'] = fotw_pd['properties'].apply(lambda p: p['lat'])

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].scatter(pix_df['lon'], pix_df['lat'], s=1, c='seagreen', alpha=0.3)
        axes[0].set_title(f'Current: {len(pix_df):,} sampled pixels\n(ESA WorldCover cropland)')
        axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

        axes[1].scatter(fotw_pd['lon'], fotw_pd['lat'], s=3, c='#e07b39', alpha=0.6)
        axes[1].set_title(f'FoTW: {n_high:,} fields (conf ≥ 0.70)\n(sample of 500 shown)')
        axes[1].set_xlabel('Longitude')

        plt.suptitle('Pixel approach vs Fields of the World — zone 000099_initial', fontsize=10)
        plt.tight_layout()
        plt.savefig('temp_images/pixels_vs_fotw.png', dpi=150)
        plt.show()
    except Exception as e:
        print(f'Could not render FoTW comparison: {e}')
else:
    print('FoTW not loaded — run Cell 11 first and confirm access')

---
## 6. Phase Two Planning — Inundation Comparison & Decision Point

### What we need to decide before building Phase Two

**Decision A — Spatial unit: pixels vs fields?**
- If FoTW has good coverage in zone 000099_initial and neighbouring zones → move to fields
- If coverage is sparse → stay with pixels, but add field-boundary-aware sampling

**Decision B — Ground truth source: our Otsu labels vs external?**
- Option 1: Use our existing SAR Otsu labels (fast, consistent, but noisy)
- Option 2: Cross-validate with Copernicus EMS for 2022/2025 events (slower, more reliable)
- Option 3: Use JRC GSW monthly maps as optical anchor for dry-season events

**Decision C — Inundation duration: binary vs binned?**
- Binary (flooded/not): simple, consistent with current labels
- Binned duration (0-4 / 4-8 / 8-16 / 16-32 days): more informative for agricultural impact

Run the cells below to compare our inundation estimates against external sources for the two largest events.

In [ ]:
# CELL 14 — Compare our flood % vs JRC GSW and/or Copernicus EMS (where available)
# For events in dry season (Oct–Mar), JRC GSW monthly composite is usable
# For 2022 Aug and 2025 Aug Pakistan floods, Copernicus EMS maps are available

if len(zone_df) == 0:
    print('No zone data — run Cell 5 first')
else:
    success = zone_df[zone_df['status'] == 'SUCCESS'].copy()
    success['flood_start'] = pd.to_datetime(success['flood_start'])

    # JRC GSW — check monthly water occurrence around each event
    lat_c = pix_df['lat'].mean() if len(pix_df) > 0 else 31.0
    lon_c = pix_df['lon'].mean() if len(pix_df) > 0 else 72.5
    buf = 0.04
    zone_bbox = ee.Geometry.BBox(lon_c - buf, lat_c - buf, lon_c + buf, lat_c + buf)

    jrc_results = []
    for _, ev in success.iterrows():
        y = ev['flood_start'].year
        m = ev['flood_start'].month
        # JRC GSW monthly: band 'water' = 2 (water), 1 (land), 0 (no data)
        jrc = (
            ee.ImageCollection('JRC/GSW1_4/MonthlyHistory')
            .filter(ee.Filter.calendarRange(y, y, 'year'))
            .filter(ee.Filter.calendarRange(m, m, 'month'))
            .first()
        )
        if jrc is None:
            jrc_results.append({'flood_start': ev['flood_start'], 'jrc_pct': None})
            continue
        try:
            stats = jrc.eq(2).reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=zone_bbox,
                scale=30
            ).getInfo()
            pct = (stats.get('water') or 0) * 100
        except Exception:
            pct = None
        jrc_results.append({'flood_start': ev['flood_start'], 'jrc_pct': pct})

    jrc_df = pd.DataFrame(jrc_results)
    compare = success[['flood_start','max_category','percentage_flooded']].merge(
        jrc_df, on='flood_start', how='left'
    )
    compare.columns = ['Flood start','Category','Our SAR Otsu %','JRC GSW %']
    print(compare.to_string(index=False))
    print()
    print('Note: JRC GSW is optical (Landsat 30m). Monsoon months will show NaN due to cloud cover.')
    print('For Aug 2022/2025 events, download Copernicus EMS maps from:')
    print('  https://emergency.copernicus.eu/mapping/list-of-components/EMSR')

In [ ]:
# CELL 15 — Decision checkpoint
# Run this cell to print the decision summary before proceeding to Phase Two implementation

print('=' * 60)
print('PHASE TWO DECISION CHECKPOINT')
print('=' * 60)
print()
print('A) Spatial unit:')
print('   [ ] Pixels (keep current approach, faster to train)')
print('   [ ] Fields of the World (more interpretable, needs FoTW coverage check)')
print()
print('B) Ground truth:')
print('   [ ] SAR Otsu labels only (fast, consistent)')
print('   [ ] SAR Otsu + Copernicus EMS cross-validation (more reliable for top events)')
print('   [ ] SAR Otsu + JRC GSW for dry-season events')
print()
print('C) Label type:')
print('   [ ] Binary flooded/not')
print('   [ ] Binned inundation duration (0-4 / 4-8 / 8-16 / 16-32 days)')
print()
print('D) Additional rivers for Phase Two:')
print('   [ ] Chenab only (zone 000099_initial, current data)')
print('   [ ] Indus + Chenab (need to run ingest_s1_flood_pixels for Indus zones)')
print('   [ ] All 10 zones (full dataset, more compute time)')
print()
print('Record your choices above and send back — will use these to build')
print('the Phase Two implementation brief.')

---
## 7. Export Phase One Summary PDF

In [ ]:
# CELL 16 — Compile Phase One summary PDF
import os
os.makedirs('reports', exist_ok=True)

PDF_PATH = 'reports/phase_one_summary.pdf'

with PdfPages(PDF_PATH) as pdf:

    # --- Page 1: Title & Status ---
    fig = plt.figure(figsize=(11, 8.5))
    fig.patch.set_facecolor('white')
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_axis_off()

    ax.text(0.5, 0.92, 'Foundation Flood — Phase One Summary',
            ha='center', fontsize=18, fontweight='bold', transform=ax.transAxes)
    ax.text(0.5, 0.87, f'Generated: {datetime.now().strftime("%Y-%m-%d")} | Zone: 000099_initial (Chenab River, Pakistan)',
            ha='center', fontsize=10, color='#555', transform=ax.transAxes)

    # Status table
    table_data = [[r['Table'], f"{r['Rows']:,}", r['Status']] for _, r in status_df.iterrows()]
    col_labels = ['Table', 'Rows', 'Status']
    tbl = ax.table(
        cellText=table_data,
        colLabels=col_labels,
        cellLoc='left',
        loc='center',
        bbox=[0.05, 0.45, 0.9, 0.38]
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor('#2c5f8a')
            cell.set_text_props(color='white', fontweight='bold')
        elif r % 2 == 0:
            cell.set_facecolor('#eef4fb')

    ax.text(0.5, 0.40,
            'Scope: 10 study zones · ~49,500 agricultural pixels · 64 confirmed flood events (2015–2025)',
            ha='center', fontsize=9, color='#333', transform=ax.transAxes)

    ax.text(0.5, 0.34, 'Pipeline',
            ha='center', fontsize=13, fontweight='bold', transform=ax.transAxes)
    steps = [
        '1. dfo-ingest      — Discharge + flood events from DFO  ✓ complete',
        '2. define-zones    — River centreline + 3×3 km study zones   ✓ complete',
        '3. flood-calc      — Otsu SAR thresholding → zone_flood_analysis  ✓ complete',
        '4. select-pixels   — ESA WorldCover cropland pixels + terrain   ✓ complete',
        '5. s1-flood-pixels — Per-pixel S1 VV/VH + flood label   ⟳ running / ready to run',
    ]
    for i, s in enumerate(steps):
        ax.text(0.08, 0.28 - i * 0.05, s, fontsize=8.5, fontfamily='monospace',
                color='#222', transform=ax.transAxes)

    pdf.savefig(fig)
    plt.close(fig)

    # --- Page 2: Flood events bar chart ---
    fig, ax = plt.subplots(figsize=(11, 7))
    success_ev = zone_df[zone_df['status'] == 'SUCCESS'].copy()
    success_ev['flood_start'] = pd.to_datetime(success_ev['flood_start'])
    colors_cat = {1: '#fee8c8', 2: '#fdbb84', 3: '#fc8d59', 4: '#e34a33', 5: '#b30000'}
    ax.bar(
        range(len(success_ev)),
        success_ev['percentage_flooded'],
        color=[colors_cat.get(int(c), '#aaa') for c in success_ev['max_category']]
    )
    ax.set_xticks(range(len(success_ev)))
    ax.set_xticklabels(
        [f"{r['flood_start'].strftime('%b %y')}\ncat{int(r['max_category'])}" for _, r in success_ev.iterrows()],
        fontsize=7
    )
    ax.set_ylabel('% zone area flooded (SAR Otsu)')
    ax.set_title('Flood extent per event — zone 000099_initial (Chenab River)')
    patches = [mpatches.Patch(color=colors_cat[c], label=f'Category {c}') for c in range(1, 6)]
    ax.legend(handles=patches, loc='upper left')
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

    # --- Page 3: Terrain distributions ---
    if len(pix_df) > 0:
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].hist(pix_df['elevation'].dropna(), bins=40, color='steelblue', edgecolor='white')
        axes[0].set_xlabel('Elevation (m)'); axes[0].set_title('Elevation')
        axes[1].hist(pix_df['slope'].dropna(), bins=40, color='darkorange', edgecolor='white')
        axes[1].set_xlabel('Slope (°)'); axes[1].set_title('Slope')
        axes[2].hist(pix_df['dist_to_river_m'].dropna(), bins=40, color='seagreen', edgecolor='white')
        axes[2].set_xlabel('Distance to river (m)'); axes[2].set_title('Distance to river')
        plt.suptitle(f'pixels_static terrain metrics — {len(pix_df):,} pixels — zone 000099_initial')
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

    # --- Page 4: Flood event maps (if saved) ---
    map_img_path = 'temp_images/flood_events_map.png'
    if os.path.exists(map_img_path):
        fig = plt.figure(figsize=(11, 8.5))
        img = Image.open(map_img_path)
        plt.imshow(img)
        plt.axis('off')
        plt.title('Flood event maps — agricultural pixels (blue=flooded, red=unflooded)', pad=10)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

    # --- Page 5: Method comparison text ---
    fig = plt.figure(figsize=(11, 8.5))
    ax = fig.add_axes([0.05, 0.05, 0.9, 0.9])
    ax.set_axis_off()
    ax.text(0.5, 0.97, 'Method Comparison & Phase Two Decisions',
            ha='center', fontsize=14, fontweight='bold', transform=ax.transAxes)

    text = """
CURRENT METHOD — SAR Otsu Thresholding
  Strengths: cloud-penetrating, unsupervised, GEE scalable
  Weaknesses: single acquisition per event, noise from bare soil, no duration signal

ALTERNATIVES REVIEWED
  JRC GSW (Landsat 30m)         — cloud-blocked during monsoon; dry-season cross-check only
  Copernicus EMS                — best ground truth for 2022/2025 major events; sparse otherwise
  AI4GOOD FloodAI (S1+S2 DL)   — higher accuracy but needs GPU; not accessible at scale
  Google Flood Hub              — real-time only; no historical per-pixel labels

PIXEL vs FIELD APPROACH
  Pixels: simple, fast to train, 49k samples, no boundary context
  Fields of the World: field-level output, inundation %, duration — needs FoTW coverage check

DECISION POINT (fill in before Phase Two)
  A. Spatial unit:      [ ] pixels   [ ] FoTW fields
  B. Ground truth:      [ ] SAR Otsu only   [ ] + Copernicus EMS   [ ] + JRC GSW
  C. Label type:        [ ] binary   [ ] binned duration
  D. River scope:       [ ] Chenab only   [ ] Indus+Chenab   [ ] All 10 zones
"""
    ax.text(0.03, 0.90, text, fontsize=9, fontfamily='monospace',
            va='top', transform=ax.transAxes, color='#222')
    pdf.savefig(fig)
    plt.close(fig)

    # Metadata
    d = pdf.infodict()
    d['Title'] = 'Foundation Flood Phase One Summary'
    d['Author'] = 'Samuel Oswald'
    d['Subject'] = 'SAR flood mapping, Pakistan, Chenab River'
    d['CreationDate'] = datetime.now()

print(f'PDF saved: {PDF_PATH}')
print(f'Pages: 5 (status, flood events, terrain, maps, method comparison)')

In [ ]:
# CELL 17 — Download PDF from Colab
from google.colab import files
if os.path.exists(PDF_PATH):
    files.download(PDF_PATH)
    print(f'Downloading {PDF_PATH}')
else:
    print(f'PDF not found at {PDF_PATH} — run Cell 16 first')